# Parte 0 — Tres fallas que no fallan

MiniLM local (tope 128) y Qdrant en memoria: sin VPN ni Docker.

In [1]:
import os
import sys

os.environ["EMBEDDING_BACKEND"] = "local"
os.environ["QDRANT_URL"] = ":memory:"
sys.path.append("..")

## 0.a — El PDF que se indexa vacío

In [2]:
!cd .. && CORPUS_DIR=ejemplos python rag_pipeline.py "¿cuántos días de vacaciones puedo transferir?"

Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 9718.31it/s]


embeddings: local · sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 · tope 128 tokens · Qdrant :memory:
AVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s). ¿Escaneado o fotografiado? NO se indexa: sin texto no hay nada que recuperar (OCR aparte).
ingesta: nimbus_gastos.md: 1 página(s), 294 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_remoto.md: 1 página(s), 296 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_vacaciones.md: 1 página(s), 363 caracteres útiles → 1 fragmento(s)


indexados 3 fragmentos de 3 documento(s)


  0.681  nimbus_vacaciones.md / nimbus_vacaciones-0000  # Política de vacaciones de NimbusSoft Los empleados a tiempo completo acumulan …
  0.502  nimbus_remoto.md / nimbus_remoto-0000  # Política de trabajo remoto de NimbusSoft El trabajo remoto está permitido hast…
  0.485  nimbus_gastos.md / nimbus_gastos-0000  # Política de reembolso de gastos de NimbusSoft Los gastos de viaje se reembolsa…



[ollama:qwen3:1.7b] abstuvo=False
El corpus indica que los días no utilizados pueden transferirse al año siguiente hasta un máximo de 5 días.  
Respuesta: 5 días.


**Pregunta:** en `ingestion.py`, ¿qué comprueba `load_corpus` para no indexar el PDF escaneado y qué produciría sin esa comprobación con ese mismo archivo? (dos frases)

`load_corpus` cuenta las letras y números de cada documento y, si tiene menos de 200, no lo indexa y muestra un aviso. Sin esa revisión, el PDF escaneado se guardaba como un fragmento que solo dice `[page=1] [page=2]`, sin ningún error, y parecía que el documento estaba en el índice aunque no tenía nada.

## 0.b — El fragmento que se corta a 128

In [3]:
from pathlib import Path

import numpy as np

from rag_pipeline import RagPipeline

pipeline = RagPipeline(collection="parte0")
tok = pipeline.encoder.tokenizer
tope = pipeline.max_seq_tokens

/home/harold/Documents/maestria/primer semestre/IA generativa y agentes/semana 2/taller 2/taller02-rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13291.08it/s]

In [4]:
pipeline.ingest(Path("../ejemplos"), chunk_tokens=900);

ingesta: nimbus_gastos.md: 1 página(s), 294 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_remoto.md: 1 página(s), 296 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_vacaciones.md: 1 página(s), 363 caracteres útiles → 1 fragmento(s)


AVISO ingesta: chunk_tokens=900 supera el tope del modelo (128 tokens): lo que pase de 126 tokens no llega al índice —el MiniLM lo trunca en silencio; la H200 rechaza la petición—.
AVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s). ¿Escaneado o fotografiado? NO se indexa: sin texto no hay nada que recuperar (OCR aparte).


In [5]:
texto = " ".join(Path("../README.md").read_text(encoding="utf-8").split()[:900])
ids = tok(texto, add_special_tokens=True, truncation=False)["input_ids"]
recorte = tok.decode(ids[1:tope - 1])

v_todo, v_recorte = pipeline.encoder.encode([texto, recorte])
print(f"palabras: {len(texto.split())}")
print(f"tokens del texto: {len(ids)} · max_seq_length del modelo: {tope}")
print(f"coseno(texto entero, recortado a {tope} tokens) = {float(v_todo @ v_recorte):.4f}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1864 > 128). Running this sequence through the model will result in indexing errors


palabras: 900
tokens del texto: 1864 · max_seq_length del modelo: 128
coseno(texto entero, recortado a 128 tokens) = 1.0000


**Pregunta:** ¿qué `chunk_tokens` vas a usar en la Parte 1? Justifícalo en una frase contra el tope de bge-m3 (que 8192 quepan no quiere decir que un fragmento de 8192 sea bueno).

Se usa `chunk_tokens=512` y `overlap_tokens=102` porque, aunque bge-m3 acepta hasta 8192 tokens, un fragmento tan grande junta muchos temas en un solo vector, y en los papers la respuesta a una pregunta normalmente está en uno o dos párrafos.

## 0.c — El índice que no se queja

In [6]:
pipeline.index(pipeline.ingest(Path("../ejemplos")))

preguntas = [
    "¿cuál es la política de mascotas?",
    "¿qué exige el instructivo de prácticas de campo sobre el seguro de accidentes?",
]
for pregunta in preguntas:
    print(pregunta)
    for hit in pipeline.retrieve(pregunta, top_k=3):
        print(f"  {hit['score']:.3f}  {hit['document']}  {hit['text'][:60]}")
    print()

AVISO ingesta: instructivo_escaneado.pdf: 0 caracteres útiles en 2 página(s). ¿Escaneado o fotografiado? NO se indexa: sin texto no hay nada que recuperar (OCR aparte).


ingesta: nimbus_gastos.md: 1 página(s), 294 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_remoto.md: 1 página(s), 296 caracteres útiles → 1 fragmento(s)
ingesta: nimbus_vacaciones.md: 1 página(s), 363 caracteres útiles → 1 fragmento(s)
¿cuál es la política de mascotas?


  0.058  nimbus_gastos.md  # Política de reembolso de gastos de NimbusSoft Los gastos d
  0.032  nimbus_remoto.md  # Política de trabajo remoto de NimbusSoft El trabajo remoto
  -0.033  nimbus_vacaciones.md  # Política de vacaciones de NimbusSoft Los empleados a tiemp

¿qué exige el instructivo de prácticas de campo sobre el seguro de accidentes?
  0.229  nimbus_remoto.md  # Política de trabajo remoto de NimbusSoft El trabajo remoto
  0.171  nimbus_gastos.md  # Política de reembolso de gastos de NimbusSoft Los gastos d
  0.099  nimbus_vacaciones.md  # Política de vacaciones de NimbusSoft Los empleados a tiemp



**Pregunta:** ¿por qué ningún Hit Rate puede detectar esto? (una frase)

Ningún Hit Rate detecta esto porque solo revisa si el documento correcto sale entre los resultados, y estas preguntas no tienen documento correcto aunque el buscador igual devuelve tres vecinos sin error, por eso el golden set lleva preguntas negativas en las que se mide si el sistema se abstiene.